In [1]:
# Frontier prediction: the update rules on the mixed-family run
# (data/frontier): n = 64 agents on ONE random graph J0 — agents 0-31
# GPT-5.6-sol, 32-63 DeepSeek-V4-Flash (personas tiled, agent i = persona
# i % 32), k = 1 spin sample, 8 steps, 1 episode per question. Fit on the
# train questions, score the test questions; with a single J there is no
# seen/fresh graph axis, so columns report model families instead.
import json
from pathlib import Path

import numpy as np
import pandas as pd

from utils import (S_PREV, FIELD, D_POS, D_NEG, BANK_DIR, Logistic, pca3,
                   two_stage_fit, discrete_rollout, fcba, raw_acc)

DATA = Path("..") / "data"
PAIR = sorted(p for p in (DATA / "frontier").iterdir() if p.is_dir())[0]
REGIMES = ["subjective", "objective"]
T = 8

META = json.load(open(PAIR / "objective_energy" / "manifest.json"))["meta"]
N = META["num_agents"]
AGENT_MODELS = np.array(META["agent_models"])
LABELS = {"openai/gpt-5.6-sol": "GPT-5.6-sol",
          "deepseek/deepseek-v4-flash-0731": "DeepSeek-V4-Flash"}
FAMS = {"All": np.ones(N, bool)}
FAMS.update({LABELS.get(m, m.split("/")[-1]): AGENT_MODELS == m
             for m in dict.fromkeys(META["agent_models"])})

# question banks (embeddings, splits, truth) and persona PCA features, as in
# utils.load_data but without the n = 32 runs
banks, P = {}, {}
for regime, sub in BANK_DIR.items():
    bank = {}
    for split in ("train", "test"):
        for line in (DATA / sub / f"{split}.jsonl").read_text().splitlines():
            q = json.loads(line)
            bank[q["question"]] = {
                "split": split, "qid": q["qid"],
                "q": np.array(q["embedding_pca10"][:3]),
                "truth": {"A": 1, "B": -1}.get(q.get("answer"), 0)}
    banks[regime] = bank
    items = sorted(json.load(open(DATA / sub / "persona_embeddings.json"))
                   ["items"], key=lambda it: it["idx"])
    P[regime] = pca3(np.array([it["embedding"] for it in items], dtype=float))

runs = {r: [json.load(open(p))
            for p in sorted((PAIR / f"{r}_energy").glob("*__J0__*.json"))]
        for r in REGIMES}
print(PAIR.name, "|", {r: len(v) for r, v in runs.items()},
      "| families:", {f: int(m.sum()) for f, m in FAMS.items() if f != "All"})

gpt-5.6-sol__deepseek-v4-flash-0731 | {'subjective': 20, 'objective': 40} | families: {'GPT-5.6-sol': 32, 'DeepSeek-V4-Flash': 32}


In [2]:
# x (E, 8, 64, 19) = [s_prev | bias p q p⊗q | drive_pos drive_neg] per
# (transition, agent); y (E, 8, 64) = the next spin s(t+1) (0 = unparsed).
# Same features as 4_prediction, with the 32 persona features tiled over 64.
def build_xy(regime):
    Pn = P[regime][np.arange(N) % len(P[regime])]
    ep = {k: [] for k in ("phi", "J", "spins", "split", "qid", "tau")}
    for r in runs[regime]:
        info = banks[regime][r["statement"]]
        q = info["q"]
        pq = (Pn[:, :, None] * q[None, None, :]).reshape(N, -1)
        ep["phi"].append(np.concatenate(
            [np.ones((N, 1)), Pn, np.tile(q, (N, 1)), pq], axis=1))
        ep["J"].append(np.array(r["J"], dtype=float))
        ep["spins"].append(np.array(r["spins_history"], dtype=float))
        ep["split"].append(info["split"])
        ep["qid"].append(info["qid"])
        ep["tau"].append(float(info["truth"]))
    ep = {k: np.array(v) for k, v in ep.items()}
    S, phi = ep["spins"], ep["phi"]
    s_in = S[:, :-1]
    pos = np.einsum("eij,etj->eti", np.maximum(ep["J"], 0), s_in)
    neg = np.einsum("eij,etj->eti", np.minimum(ep["J"], 0), s_in)
    x = np.concatenate([s_in[..., None],
                        np.broadcast_to(phi[:, None],
                                        (len(S), T) + phi.shape[1:]),
                        pos[..., None], neg[..., None]], axis=-1)
    return x, S[:, 1:].astype(int), ep

x, y, ep = build_xy("objective")
print("x", x.shape, " y", y.shape)

x (40, 8, 64, 19)  y (40, 8, 64)


In [3]:
# the table methods; each returns (onestep, rollout) test predictions
# (identical to 4_prediction, with the agent count n = 64 not hard-coded)
def predict_all(x_tr, y_tr, x_te, J_te):
    n = x_te.shape[2]
    rows = x_tr.reshape(-1, x_tr.shape[-1])          # pooled train transitions
    parsed = y_tr.reshape(-1) != 0                   # unparsed targets excluded
    y01 = (y_tr.reshape(-1)[parsed] > 0).astype(float)
    E, s0, phi = len(x_te), x_te[:, 0, :, S_PREV], x_te[:, 0, :, FIELD]
    out = {}

    c = 1 if y01.mean() >= 0.5 else -1
    const = np.full((E, T, n), c, dtype=int)
    out["Majority Class"] = (const, const)

    # persistence: no change; rollout frozen at s(0)
    out["Persistence"] = (x_te[..., S_PREV].astype(int),
                          np.repeat(s0[:, None].astype(int), T, axis=1))

    # interaction-free: logistic on the static field only (state-independent)
    clf = Logistic().fit(rows[parsed][:, FIELD], y01)
    pred = clf.predict_spin(x_te[..., FIELD])
    out["Interaction-Free"] = (pred, pred)

    # mean-field (Curie-Weiss): [field | population mean s̄(t)]
    def with_sbar(xx):
        sbar = xx[..., S_PREV].mean(axis=-1)
        return np.concatenate([xx[..., FIELD],
                               np.broadcast_to(sbar[..., None, None],
                                               xx.shape[:-1] + (1,))], axis=-1)
    clf = Logistic().fit(with_sbar(x_tr).reshape(-1, 17)[parsed], y01)
    onestep = clf.predict_spin(with_sbar(x_te))
    s, rollout = s0.copy(), np.empty((E, T, n), dtype=int)
    for t in range(T):
        sbar = np.broadcast_to(s.mean(axis=1)[:, None, None], (E, n, 1))
        s = clf.predict_spin(np.concatenate([phi, sbar], axis=-1)).astype(float)
        rollout[:, t] = s
    out["Mean-Field (Curie-Weiss)"] = (onestep, rollout)

    # discrete update with 1 coupling [(J s)]
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    dr = (x_te[..., D_POS] + x_te[..., D_NEG])[..., None]
    onestep = clf.predict_spin(np.concatenate([x_te[..., FIELD], dr], axis=-1))
    out["Discrete Update"] = (onestep, discrete_rollout(phi, J_te, s0, clf.w, 1))

    # + 3 couplings [(J+ s) | (J- s) | (|J| s)]: collinear, so fit in two stages
    w = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                      pos - neg, parsed, y01)
    dr = np.stack([x_te[..., D_POS], x_te[..., D_NEG],
                   x_te[..., D_POS] - x_te[..., D_NEG]], axis=-1)
    onestep = np.where(np.concatenate([x_te[..., FIELD], dr], axis=-1) @ w > 0, 1, -1)
    out["+ 3 Couplings"] = (onestep, discrete_rollout(phi, J_te, s0, w, 3))
    return out

In [4]:
# metrics from utils: fcba is flip-and-class balanced accuracy
#, the unweighted mean over (flip vs stay) x (next spin +-1);
# raw_acc is plain accuracy. Majority Class only appears under raw accuracy.
METRIC_ROWS = {
    "balanced": ["Persistence", "Interaction-Free", "Mean-Field (Curie-Weiss)",
                 "Discrete Update", "+ 3 Couplings"],
    "raw": ["Majority Class", "Persistence", "Interaction-Free",
            "Mean-Field (Curie-Weiss)", "Discrete Update", "+ 3 Couplings"]}
METRIC_NAMES = {"balanced": "flip-and-class balanced accuracy",
                "raw": "raw accuracy"}

In [5]:
# fit on the train questions, score on the test questions (both on the
# single J0); scores per model family by slicing the agent axis
results = {}
for regime in REGIMES:
    x, y, ep = build_xy(regime)
    train, test = ep["split"] == "train", ep["split"] == "test"
    preds = predict_all(x[train], y[train], x[test], ep["J"][test])
    y_te, s0 = y[test], x[test][:, 0, :, S_PREV]
    mask = np.ones(int(test.sum()), dtype=bool)
    score = {"balanced": lambda p, f: fcba(p[..., f], y_te[..., f], s0[:, f], mask),
             "raw": lambda p, f: raw_acc(p[..., f], y_te[..., f], mask)}
    for method, (onestep, rollout) in preds.items():
        for metric, fn in score.items():
            for fam, fidx in FAMS.items():
                results[metric, regime, fam, method] = (fn(onestep, fidx),
                                                        fn(rollout, fidx))
    print(f"{regime:10s}  train {train.sum()}, test {test.sum()} episodes")

subjective  train 10, test 10 episodes
objective   train 20, test 20 episodes


In [6]:
# the tables: one-step (rollout) accuracy per cell, per metric
def cell(metric, regime, fam, method):
    o, r = results[metric, regime, fam, method]
    return f"{o:.1f} ({r:.1f})"

for metric in ("balanced", "raw"):
    print(f"=== {METRIC_NAMES[metric].capitalize()} — one-step (rollout) ===")
    display(pd.DataFrame({(rg.capitalize(), fam):
                          {m: cell(metric, rg, fam, m)
                           for m in METRIC_ROWS[metric]}
                          for rg in REGIMES for fam in FAMS}
                         ).reindex(METRIC_ROWS[metric]))

=== Flip-and-class balanced accuracy — one-step (rollout) ===


Subjective                                 \
                                  All  GPT-5.6-sol DeepSeek-V4-Flash   
Persistence               50.0 (63.3)  50.0 (63.2)       50.0 (63.5)   
Interaction-Free          51.7 (51.7)  52.2 (52.2)       51.1 (51.1)   
Mean-Field (Curie-Weiss)  57.3 (56.3)  57.0 (56.7)       57.6 (56.0)   
Discrete Update           63.3 (59.0)  65.5 (60.4)       61.2 (57.4)   
+ 3 Couplings             81.9 (70.4)  83.9 (71.1)       79.8 (69.7)   

                            Objective                                 
                                  All  GPT-5.6-sol DeepSeek-V4-Flash  
Persistence               50.0 (57.7)  50.0 (62.8)       50.0 (53.2)  
Interaction-Free          48.4 (48.4)  49.2 (49.2)       48.2 (48.2)  
Mean-Field (Curie-Weiss)  75.0 (70.9)  72.2 (68.7)       76.2 (71.5)  
Discrete Update           48.4 (47.9)  49.0 (48.8)       48.2 (47.5)  
+ 3 Couplings             80.0 (67.4)  74.5 (64.6)       83.5 (69.2)

=== Raw accuracy — one-step (rollout) ===


Subjective                                 \
                                  All  GPT-5.6-sol DeepSeek-V4-Flash   
Majority Class            56.3 (56.3)  54.5 (54.5)       58.1 (58.1)   
Persistence               75.0 (73.2)  75.0 (72.8)       75.0 (73.6)   
Interaction-Free          55.1 (55.1)  54.8 (54.8)       55.4 (55.4)   
Mean-Field (Curie-Weiss)  62.4 (61.0)  61.6 (60.7)       63.2 (61.4)   
Discrete Update           65.7 (61.9)  67.6 (62.9)       63.7 (60.9)   
+ 3 Couplings             82.7 (71.8)  85.0 (72.5)       80.3 (71.1)   

                            Objective                                 
                                  All  GPT-5.6-sol DeepSeek-V4-Flash  
Majority Class            50.3 (50.3)  51.7 (51.7)       49.0 (49.0)  
Persistence               88.3 (73.7)  91.3 (83.1)       85.2 (64.3)  
Interaction-Free          47.8 (47.8)  48.2 (48.2)       47.4 (47.4)  
Mean-Field (Curie-Weiss)  91.6 (84.6)  93.0 (86.3)       90.1 (82.9)  
Discrete Update           47.2 (47.2)  47.7 (47.9)       46.7 (46.5)  
+ 3 Couplings             90.2 (75.0)  89.6 (75.8)       90.8 (74.1)

In [7]:
# the LaTeX tables grouped by model and regime, one per metric (best one-step per
# column in bold); columns are model families, regimes are row blocks
for metric in ("balanced", "raw"):
    rows_m = METRIC_ROWS[metric]
    fams = list(FAMS)
    nc = len(fams)
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             rf"\begin{{tabular}}{{l {'c' * nc}}}", r"\toprule",
             "Method & " + " & ".join(fams) + r" \\"]
    for regime in REGIMES:
        lines += [r"\midrule",
                  rf"\multicolumn{{{nc + 1}}}{{l}}{{\textit{{{regime.capitalize()} Questions}}}} \\",
                  r"\midrule"]
        for method in rows_m:
            cs = []
            for fam in fams:
                o, r = results[metric, regime, fam, method]
                best = max(results[metric, regime, fam, m][0] for m in rows_m)
                txt = f"{o:.1f} ({r:.1f})"
                cs.append(rf"\textbf{{{txt}}}" if o == best else txt)
            lines.append(f"{method} & " + " & ".join(cs) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}",
              rf"\caption{{\textit{{Frontier prediction.}} "
              f"{METRIC_NAMES[metric].capitalize()} of predictions on the "
              "held-out test questions of the mixed-family run (n = 64, "
              "one graph): GPT-5.6-sol and DeepSeek-V4-Flash agents on a "
              "single signed random graph.}", r"\end{table}"]
    print("\n".join(lines) + "\n")

\begin{table}[t]
\centering
\small
\begin{tabular}{l ccc}
\toprule
Method & All & GPT-5.6-sol & DeepSeek-V4-Flash \\
\midrule
\multicolumn{4}{l}{\textit{Subjective Questions}} \\
\midrule
Persistence & 50.0 (63.3) & 50.0 (63.2) & 50.0 (63.5) \\
Interaction-Free & 51.7 (51.7) & 52.2 (52.2) & 51.1 (51.1) \\
Mean-Field (Curie-Weiss) & 57.3 (56.3) & 57.0 (56.7) & 57.6 (56.0) \\
Discrete Update & 63.3 (59.0) & 65.5 (60.4) & 61.2 (57.4) \\
+ 3 Couplings & \textbf{81.9 (70.4)} & \textbf{83.9 (71.1)} & \textbf{79.8 (69.7)} \\
\midrule
\multicolumn{4}{l}{\textit{Objective Questions}} \\
\midrule
Persistence & 50.0 (57.7) & 50.0 (62.8) & 50.0 (53.2) \\
Interaction-Free & 48.4 (48.4) & 49.2 (49.2) & 48.2 (48.2) \\
Mean-Field (Curie-Weiss) & 75.0 (70.9) & 72.2 (68.7) & 76.2 (71.5) \\
Discrete Update & 48.4 (47.9) & 49.0 (48.8) & 48.2 (47.5) \\
+ 3 Couplings & \textbf{80.0 (67.4)} & \textbf{74.5 (64.6)} & \textbf{83.5 (69.2)} \\
\bottomrule
\end{tabular}
\caption{\textit{Frontier prediction.} Flip-

In [8]:
# fitted couplings on the train questions (families pooled): the discrete
# update at one / three couplings; three is collinear, so beta_0 (the
# unsigned neighborhood (|J| s)) is fit first and the signed betas second.
# Not written to couplings.json — that file belongs to the n = 32 study.
BETA_LABELS = {"one": ["beta"], "three": ["beta_pos", "beta_neg", "beta_0"]}
BETA_TEX = {"beta": "β", "beta_pos": "β⁺", "beta_neg": "β⁻", "beta_0": "β₀"}

couplings = {}
for regime in REGIMES:
    x, y, ep = build_xy(regime)
    train = ep["split"] == "train"
    rows = x[train].reshape(-1, x.shape[-1])
    parsed = y[train].reshape(-1) != 0
    y01 = (y[train].reshape(-1)[parsed] > 0).astype(float)
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    w3 = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                       pos - neg, parsed, y01)
    couplings[regime] = {
        "one": {"beta": float(clf.w[16])},
        "three": dict(zip(BETA_LABELS["three"], w3[16:].tolist()))}

pd.DataFrame({regime: {(tier.capitalize(), BETA_TEX[lab]): round(v, 2)
                       for tier in ("one", "three")
                       for lab, v in couplings[regime][tier].items()}
              for regime in REGIMES})

subjective  objective
One   β         0.75       0.07
Three β⁺        1.95       0.83
      β⁻        0.76       0.41
      β₀        0.63       0.99